In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure data routines
import pandas as pd

# Import the CRUD Python module
from animal_shelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# MongoDB username and password
username = "aacuser"
password ="Jairofr03262004#"

# Connect to the database through the CRUD module
db = AnimalShelter(username, password)

# Read all animal records for the initial dashboard
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId because Dash cannot display it
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Use the exact name of your logo file
image_filename = "Grazioso Salvare Logo.png"

# Open and encode the logo
encoded_image = base64.b64encode(
    open(image_filename, "rb").read()
).decode()


app.layout = html.Div([

    # Logo linked to the SNHU website
    html.Center([
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image),
                style={
                    "height": "120px",
                    "marginBottom": "10px"
                }
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),

        html.H1("Grazioso Salvare Rescue Dashboard"),

        # Unique identifier
        html.H3("Created by Jairo Figueroa")
    ]),

    html.Hr(),

    # Interactive filter options
    html.Div([
        html.H3("Select a Rescue Type"),

        dcc.RadioItems(
            id="filter-type",

            options=[
                {
                    "label": " Water Rescue",
                    "value": "Water Rescue"
                },
                {
                    "label": " Mountain or Wilderness Rescue",
                    "value": "Mountain Rescue"
                },
                {
                    "label": " Disaster or Individual Tracking",
                    "value": "Disaster Rescue"
                },
                {
                    "label": " Reset",
                    "value": "Reset"
                }
            ],

            value="Reset",

            labelStyle={
                "display": "inline-block",
                "marginRight": "25px"
            }
        )
    ], style={"textAlign": "center"}),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id="datatable-id",

        columns=[
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in df.columns
        ],

        data=df.to_dict("records"),

        row_selectable="single",
        selected_rows=[0],

        page_action="native",
        page_current=0,
        page_size=10,

        sort_action="native",
        sort_mode="multi",

        filter_action="native",

        style_table={
            "overflowX": "auto",
            "overflowY": "auto",
            "maxHeight": "500px"
        },

        style_header={
            "fontWeight": "bold",
            "textAlign": "center"
        },

        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "200px",
            "whiteSpace": "normal",
            "padding": "5px"
        }
    ),

    html.Br(),
    html.Hr(),

    # Pie chart and map displayed side-by-side
    html.Div(
        className="row",

        style={
            "display": "flex",
            "flexWrap": "wrap"
        },

        children=[

            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={
                    "width": "50%",
                    "minWidth": "450px"
                }
            ),

            html.Div(
                id="map-id",
                className="col s12 m6",
                style={
                    "width": "50%",
                    "minWidth": "450px"
                }
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "columns"),
        Output("datatable-id", "selected_rows")
    ],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    # Water Rescue query
    if filter_type == "Water Rescue":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Mountain or Wilderness Rescue query
    elif filter_type == "Mountain Rescue":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Disaster or Individual Tracking query
    elif filter_type == "Disaster Rescue":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # Reset query
    else:
        query = {}

    # Retrieve matching records through the CRUD module
    filtered_df = pd.DataFrame.from_records(db.read(query))

    # Remove MongoDB ObjectId
    if "_id" in filtered_df.columns:
        filtered_df.drop(columns=["_id"], inplace=True)

    # Keep the table structure if no records are returned
    if filtered_df.empty:
        filtered_df = pd.DataFrame(columns=df.columns)

    columns = [
        {
            "name": column,
            "id": column,
            "deletable": False,
            "selectable": True
        }
        for column in filtered_df.columns
    ]

    data = filtered_df.to_dict("records")

    if len(data) > 0:
        selected_rows = [0]
    else:
        selected_rows = []

    return data, columns, selected_rows


# Display animal breeds based on the filtered table
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return html.Div([
            html.H3("Breed Distribution"),
            html.P("No animals match the selected filter.")
        ])

    dff = pd.DataFrame.from_dict(viewData)

    # Count the breeds and display the 10 most common
    breed_counts = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(10)
        .reset_index()
    )

    breed_counts.columns = ["breed", "count"]

    # Create the breed distribution pie chart
    figure = px.pie(
        breed_counts,
        values="count",
        names="breed",
        title="Top 10 Preferred Animal Breeds"
    )

    # Display percentages inside the pie chart
    figure.update_traces(
        textposition="inside",
        textinfo="percent"
    )

    return [
        dcc.Graph(
            figure=figure
        )
    ]


# Highlight a selected table column
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):

    if selected_columns is None:
        return []

    return [
        {
            "if": {
                "column_id": column
            },
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ]


# Update the geolocation chart for the selected animal
@app.callback(
    Output("map-id", "children"),

    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return html.Div([
            html.H3("Animal Location"),
            html.P("No location is available.")
        ])

    dff = pd.DataFrame.from_dict(viewData)

    # Use the first row when no row is selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Prevent an invalid row number
    if row >= len(dff):
        row = 0

    latitude = dff.iloc[row]["location_lat"]
    longitude = dff.iloc[row]["location_long"]

    breed = dff.iloc[row]["breed"]
    animal_name = dff.iloc[row]["name"]

    if pd.isna(animal_name):
        animal_name = "Unnamed Animal"

    return [
        html.H3(
            "Selected Animal Location",
            style={"textAlign": "center"}
        ),

        dl.Map(
            style={
                "width": "100%",
                "height": "500px"
            },

            center=[latitude, longitude],
            zoom=10,

            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[latitude, longitude],

                    children=[
                        dl.Tooltip(str(breed)),

                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(str(animal_name)),
                            html.H4("Breed"),
                            html.P(str(breed))
                        ])
                    ]
                )
            ]
        )
    ]


# Run the dashboard
app.run_server()

Dash app running on https://scholarquarter-fiestadragon-3000.codio.io/proxy/8050/
